# Course 1 — The Numerical Engine (NumPy)

Live-coding notebook that mirrors the slide deck. Run each cell as you teach.

**Sections**
1. The ND-Array (0:00–0:30)
2. Shape & Slicing (0:30–1:00)
3. Vectorization & Broadcasting (1:00–1:30)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)   # seeded RNG — every run reproducible
np.__version__


## 1. The ND-Array

### List vs. ndarray — the memory story

In [ ]:
py_list = [1, 2, 3, 4]
arr = np.array([1, 2, 3, 4])
print(py_list, type(py_list))
print(arr, type(arr), arr.dtype)


### Why it's fast — measure it

In [ ]:
# Sum of one million ints
%timeit sum(range(1_000_000))
%timeit np.arange(1_000_000).sum()


### dtypes — one float promotes the whole array

In [ ]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1.0, 2, 3]).dtype)
print(np.array([1, 2, 3], dtype=np.float32).dtype)


### Creating arrays

In [ ]:
print(np.zeros((2, 3)))
print(np.ones((2, 3), dtype=int))
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))
print(rng.normal(size=(2, 3)))


## 2. Shape & Slicing

In [ ]:
a = np.arange(12).reshape(3, 4)
print(a)
print('shape:', a.shape, 'ndim:', a.ndim, 'size:', a.size, 'dtype:', a.dtype)


### reshape with -1 — let NumPy compute the missing dimension

In [ ]:
x = np.arange(24)
print(x.reshape(2, 3, 4).shape)
print(x.reshape(2, -1).shape)


### Transpose & axis-swap (think batch-of-images conversions)

In [ ]:
img = rng.random((224, 224, 3))   # HWC layout (e.g. matplotlib)
print('HWC:', img.shape)
print('CHW:', img.transpose(2, 0, 1).shape)   # PyTorch layout


### Slicing in N dimensions

In [ ]:
a = np.arange(20).reshape(4, 5)
print(a)
print('row 1     :', a[1])
print('col 0     :', a[:, 0])
print('2x2 block :\n', a[1:3, 2:4])
print('reversed  :\n', a[::-1])


**Gotcha:** slices are *views*, not copies.

In [ ]:
b = a[:2, :2]
b[0, 0] = -999
print(a)   # the source mutated too!


### Boolean & fancy indexing

In [ ]:
x = rng.normal(size=10)
print('x         :', x)
print('positives :', x[x > 0])
x[x < 0] = 0
print('clipped   :', x)

ix = np.array([0, 3, 5])
print('picked    :', x[ix])


## 3. Vectorization & Broadcasting

### The loop you should never write

In [ ]:
x = rng.normal(size=1_000_000)

def slow(x):
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = x[i]**2 + 3*x[i]
    return out

def fast(x):
    return x**2 + 3*x

%timeit slow(x)
%timeit fast(x)


### Reductions — learn the `axis` argument

In [ ]:
a = np.arange(12).reshape(3, 4)
print(a)
print('sum all   :', a.sum())
print('sum axis0 :', a.sum(axis=0))   # collapse rows -> per-column total
print('sum axis1 :', a.sum(axis=1))   # collapse cols -> per-row total


### Broadcasting — align shapes from the right

In [ ]:
row = np.array([1, 2, 3])
M = np.zeros((3, 3))
print(M + row)        # (3,) broadcast over (3, 3)

col = np.array([[10], [20], [30]])   # (3, 1)
print(col + row)      # (3, 1) + (3,) -> (3, 3)


### Worked example — pairwise distances without a single Python loop

In [ ]:
pts = rng.normal(size=(100, 2))
diff = pts[:, None, :] - pts[None, :, :]    # (100, 100, 2)
dist = np.sqrt((diff**2).sum(axis=-1))
print('dist matrix shape:', dist.shape)
print('symmetric? ', np.allclose(dist, dist.T))
print('zero diagonal?', np.allclose(np.diag(dist), 0))


### Recap
* ndarray is a contiguous typed buffer — that is the entire speed story.
* Reshape and transpose change *strides*, not data.
* Replace numeric `for` loops with broadcasting + reductions.